## Registering LGMS COGs as Earth Engine Assets

Created by Mel Rose; Last updated 9/12/2026

## Setup

In [3]:
#TODO: This script is behind the current model version (1.0.6) Change to cn paths after merging to updated model branch.
#TODO: Add in other variations (by gas, land state node, land use, organic soil state, combined LULUCF, etc)
#TODO: Require which gas to run from command line argument, if none supplied default to all_gases. Update in both s3 and gcs dirs.

#TODO: Update with cn patterns
#TODO: Update with making timeseries a stacked band,
#TODO: remove test from asset name

from google.cloud import storage
import ee
import json
from pprint import pprint
import google.auth
from google.auth.transport.requests import AuthorizedSession
from src.utilities import constants_and_names as cn

# --------------------------------------------------------------------------------------------------
# USER SELECTIONS
# Leave as None to run all available options.

# Options: ["vegetation", "mineral_soil", "organic_soil"]
datasets = ["vegetation"]

# Options: ["emissions", "removals", "netflux"]
fluxes = ["emissions", "removals"]

# Options: [2016, 2017, 2018, 2019, 2020]
# Note: Soil years are resolved below so that if you run [2016, 2017, 2018] it will use the 2016_2020 interval and create just a single EE asset registration for that interval
years = [2017]

print(f"Datasets selected: {datasets}")
print(f"Fluxes selected:   {fluxes}")
print(f"Years selected:    {years}")
# --------------------------------------------------------------------------------------------------

Datasets selected: ['vegetation']
Fluxes selected:   ['emissions', 'removals']
Years selected:    [2017]


## Constants

In [23]:
veg_model_version_underscore = "1_0_5"
ee_project = "landandcarbon"
gcs_bucket = "wri-lcl-lgms"

veg_asset_folder = f"vegetation/v{veg_model_version_underscore}"
min_soil_asset_folder = f"soil/mineral/v{cn.SOC_model_version_underscore}"
org_soil_asset_folder = f"soil/organic/v{cn.organic_soil_model_version_underscore}"

years = [2017]
#years = [list(range(2016, 2025))]

# Resolve soil years
mineral_soil_years = sorted({cn.organic_soil_year_map[year] for year in years})
organic_soil_years = sorted({cn.organic_soil_year_map[year] for year in years})

# TODO: Change depending on gas and/ or carbon pool selected
emissions_pattern = "gross_emissions__all_C_pools__all_gases__MgCO2e"
removals_pattern = "gross_removals__all_C_pools__MgCO2"
netflux_pattern = "net_flux__all_C_pools__all_gases__MgCO2e"

#TODO: Add mineral soil and organic soil pattern


In [24]:
credentials, _ = google.auth.default(
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/cloud-platform'
    ]
)

credentials = credentials.with_quota_project(ee_project)
ee.Initialize(credentials=credentials, project=ee_project)
session = AuthorizedSession(credentials)
storage_client = storage.Client(project=ee_project)
bucket = storage_client.get_bucket(gcs_bucket)

## Build registration specifications


In [25]:
# TODO: Remove "test" from asset_name
def interval_bounds(period):
    # Annual vegetation outputs
    if isinstance(period, int):
        return (
            f"{period}-01-01T00:00:00.000000000Z",
            f"{period + 1}-01-01T00:00:00.000000000Z",
        )

    # Multi-year soil outputs such as 2016_2020 or 2021_2024
    start_year, end_year = map(int, str(period).split("_"))
    return (
        f"{start_year}-01-01T00:00:00.000000000Z",
        f"{end_year + 1}-01-01T00:00:00.000000000Z",
    )


asset_specs = {}

def add_spec(key, dataset, flux, period, pattern, gcs_dir, ee_folder,
             version, component=None):
    filename = f"{pattern}_{period}.tif"
    gcs_uri = f"gs://{gcs_bucket}/{gcs_dir.strip('/')}/{filename}"
    asset_name = f"{pattern}_{period}_test"
    asset_id = f"projects/{ee_project}/assets/{ee_folder.strip('/')}/{asset_name}"
    start_time, end_time = interval_bounds(period)

    asset_specs[key] = {
        "dataset": dataset,
        "flux": flux,
        "component": component,
        "period": period,
        "version": version,
        "filename": filename,
        "gcs_uri": gcs_uri,
        "asset_folder": ee_folder.strip("/"),
        "asset_name": asset_name,
        "asset_id": asset_id,
        "start_time": start_time,
        "end_time": end_time,
    }


# --------------------------------------------------------------------------------------------------
# Vegetation
if "vegetation" in datasets:
    for year in years:
        if "emissions" in fluxes:
            add_spec(
                key=f"vegetation_emissions_{year}",
                dataset="vegetation",
                flux="emissions",
                period=year,
                pattern=f"{emissions_pattern}_ha_yr",
                gcs_dir=f"{veg_asset_folder}/emissions/{emissions_pattern}",
                ee_folder=f"wri_lgms/{veg_asset_folder}/emissions",
                version=f"v{veg_model_version_underscore}",
            )

        if "removals" in fluxes:
            add_spec(
                key=f"vegetation_removals_{year}",
                dataset="vegetation",
                flux="removals",
                period=year,
                pattern=f"{removals_pattern}_ha_yr",
                gcs_dir=f"{veg_asset_folder}/removals/{removals_pattern}",
                ee_folder=f"wri_lgms/{veg_asset_folder}/removals",
                version=f"v{veg_model_version_underscore}",
            )

        if "netflux" in fluxes:
            add_spec(
                key=f"vegetation_netflux_{year}",
                dataset="vegetation",
                flux="netflux",
                period=year,
                pattern=f"{netflux_pattern}_ha_yr",
                gcs_dir=f"{veg_asset_folder}/net_flux/{netflux_pattern}",
                ee_folder=f"wri_lgms/{veg_asset_folder}/net_flux",
                version=f"v{veg_model_version_underscore}",
            )


# --------------------------------------------------------------------------------------------------
# Mineral soil
# Uses the final multi-year COG interval directly for both the GCS COG filename
# and the Earth Engine asset name (e.g. 2016_2020).
mineral_configs = {
    "emissions": cn.SOC_loss_min_soil_extent_pattern,
    "removals": cn.SOC_gain_min_soil_extent_pattern,
    "netflux": cn.SOC_net_min_soil_extent_pattern,
}

if "mineral_soil" in datasets:
    for period in mineral_soil_years:
        for flux, base_pattern in mineral_configs.items():
            if flux not in fluxes:
                continue

            add_spec(
                key=f"mineral_soil_{flux}_{period}",
                dataset="mineral_soil",
                flux=flux,
                period=period,
                pattern=f"{base_pattern}_ha_yr",
                gcs_dir=f"{min_soil_asset_folder}/{base_pattern}",
                ee_folder=f"wri_lgms/{min_soil_asset_folder}/{flux}",
                version=f"v{cn.SOC_model_version_underscore}",
            )


# --------------------------------------------------------------------------------------------------
# Organic soil
# Organic soil only has emissions outputs, split into burned and drained components.
organic_configs = {
    "burned": cn.burned_organic_soils_total_pattern,
    "drained": cn.drained_organic_soils_total_pattern,
}

if "organic_soil" in datasets and "emissions" in fluxes:
    for period in organic_soil_years:
        for component, pattern in organic_configs.items():
            add_spec(
                key=f"organic_soil_emissions_{component}_{period}",
                dataset="organic_soil",
                flux="emissions",
                component=component,
                period=period,
                pattern=pattern,
                gcs_dir=f"{org_soil_asset_folder}/{pattern}",
                ee_folder=f"wri_lgms/{org_soil_asset_folder}/emissions/{component}",
                version=f"v{cn.organic_soil_model_version_underscore}",
            )

print(f"Built {len(asset_specs)} Earth Engine registration specifications.")

Built 2 Earth Engine registration specifications.


In [26]:
## Print input COG and Earth Engine asset paths
for key, spec in asset_specs.items():
    print(f"\n{key}")
    print(f"  COG:      {spec['gcs_uri']}")
    print(f"  EE asset: {spec['asset_id']}")


vegetation_emissions_2017
  COG:      gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017.tif
  EE asset: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017_test

vegetation_removals_2017
  COG:      gs://wri-lcl-lgms/vegetation/v1_0_5/removals/gross_removals__all_C_pools__MgCO2/gross_removals__all_C_pools__MgCO2_ha_yr_2017.tif
  EE asset: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/removals/gross_removals__all_C_pools__MgCO2_ha_yr_2017_test


In [27]:
## Verify that the COGs exist in GCS
missing_cogs = []

for key, spec in asset_specs.items():
    object_name = spec["gcs_uri"].replace(f"gs://{gcs_bucket}/", "", 1)
    exists = bucket.blob(object_name).exists()

    print(f"{key}: {spec['gcs_uri']} -> exists={exists}")

    if not exists:
        missing_cogs.append(spec["gcs_uri"])

print(f"\nCOGs found: {len(asset_specs) - len(missing_cogs)}/{len(asset_specs)}")
if missing_cogs:
    print("\nMissing COGs:")
    for uri in missing_cogs:
        print(f"  {uri}")

vegetation_emissions_2017: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017.tif -> exists=True
vegetation_removals_2017: gs://wri-lcl-lgms/vegetation/v1_0_5/removals/gross_removals__all_C_pools__MgCO2/gross_removals__all_C_pools__MgCO2_ha_yr_2017.tif -> exists=True

COGs found: 2/2


## Register the COGs in GEE

Each GeoTIFF is registered as its own COG-backed `ee.Image` asset. The source remains in Google Cloud Storage; Earth Engine references the external COG rather than ingesting a copy.

Information on image manifests and registering COGs as assets:
* https://developers.google.com/earth-engine/guides/image_manifest
* https://developers.google.com/earth-engine/Earth_Engine_asset_from_cloud_geotiff

In [28]:
# Create an Earth Engine folder hierarchy if it does not already exist
def ensure_ee_folder(folder_path, ee_project):
    parts = folder_path.strip("/").split("/")
    current_path = f"projects/{ee_project}/assets"

    for part in parts:
        current_path = f"{current_path}/{part}"

        try:
            ee.data.getAsset(current_path)
            print(f"Exists:  {current_path}")
        except ee.EEException:
            ee.data.createAsset({"type": "FOLDER"}, current_path)
            print(f"Created: {current_path}")

# Function to execute COG registration request
def register_cog_as_ee_asset(session, request, ee_project):
    url = f"https://earthengine.googleapis.com/v1alpha/projects/{ee_project}/image:importExternal"
    response = session.post(url=url, data=json.dumps(request))
    return json.loads(response.content)

# Make assets public
def make_ee_asset_public(asset_id):
    ee.data.setAssetAcl(
        asset_id,
        {
            "all_users_can_read": True
        }
    )
    print(f"PUBLIC: {asset_id}")

In [29]:
# Create every required Earth Engine folder once.
for folder in sorted({spec["asset_folder"] for spec in asset_specs.values()}):
    ensure_ee_folder(folder, ee_project)

Exists:  projects/landandcarbon/assets/wri_lgms
Exists:  projects/landandcarbon/assets/wri_lgms/vegetation
Exists:  projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5
Created: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/emissions
Exists:  projects/landandcarbon/assets/wri_lgms
Exists:  projects/landandcarbon/assets/wri_lgms/vegetation
Exists:  projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5
Created: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/removals


In [30]:
# TODO: Make multiband images for timeseries per dataset

# Build the Earth Engine image manifests.
requests = {}

for key, spec in asset_specs.items():
    properties = {
        "dataset": spec["dataset"],
        "flux": spec["flux"],
        "period": str(spec["period"]),
        "version": spec["version"],
    }

    if spec["component"] is not None:
        properties["component"] = spec["component"]

    # Preserve the annual 'year' property for vegetation outputs.
    if isinstance(spec["period"], int):
        properties["year"] = spec["period"]

    requests[key] = {
        "imageManifest": {
            "name": spec["asset_id"],
            "tilesets": [
                {
                    "sources": [
                        {"uris": [spec["gcs_uri"]]}
                    ]
                }
            ],
            "startTime": spec["start_time"],
            "endTime": spec["end_time"],
            "properties": properties,
        }
    }

print(f"Built {len(requests)} Earth Engine registration requests.")

Built 2 Earth Engine registration requests.


## Execute registration

This cell registers every configured COG. It skips an asset if that Earth Engine asset ID already exists, making the cell safer to rerun.

In [32]:
results = {}

for key, request in requests.items():
    asset_id = asset_specs[key]["asset_id"]

    try:
        ee.data.getAsset(asset_id)
        print(f"SKIP - already exists: {asset_id}")

        make_ee_asset_public(asset_id) # Makes existing asset public if not already set to public

        results[key] = {"status": "already_exists", "asset_id": asset_id}
        continue

    except ee.EEException:
        pass

    print(f"REGISTER: {asset_specs[key]['gcs_uri']}")
    print(f"      -> {asset_id}")

    results[key] = register_cog_as_ee_asset(session, request, ee_project)
    pprint(results[key])


    # Make newly registered asset public
    make_ee_asset_public(asset_id)

SKIP - already exists: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017_test
PUBLIC: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017_test
SKIP - already exists: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/removals/gross_removals__all_C_pools__MgCO2_ha_yr_2017_test
PUBLIC: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/removals/gross_removals__all_C_pools__MgCO2_ha_yr_2017_test


In [34]:
folders = set()

for spec in asset_specs.values():
    parent = spec["asset_id"].rsplit("/", 1)[0]
    parts = parent.split("/")
    assets_index = parts.index("assets")

    for i in range(assets_index + 2, len(parts) + 1):
        folders.add("/".join(parts[:i]))

folders = sorted(folders, key=lambda x: (x.count("/"), x))

for folder in folders:
    info = ee.data.getAsset(folder)
    ee.data.setAssetAcl(info["name"], {"all_users_can_read": True})
    print(f"PUBLIC FOLDER: {info['name']}")

PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/vegetation
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/emissions
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/removals


## Verify the registered assets


In [35]:
# Print the expected Earth Engine asset IDs
for key, spec in asset_specs.items():
    print(f"\n{key}")
    print(f"  COG:      {spec['gcs_uri']}")
    print(f"  EE asset: {spec['asset_id']}")

    try:
        info = ee.data.getAsset(spec["asset_id"])
        print(f"  EE type:  {info.get('type')}")
    except ee.EEException as exc:
        print(f"  NOT FOUND in Earth Engine: {exc}")


vegetation_emissions_2017
  COG:      gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017.tif
  EE asset: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017_test
  EE type:  IMAGE

vegetation_removals_2017
  COG:      gs://wri-lcl-lgms/vegetation/v1_0_5/removals/gross_removals__all_C_pools__MgCO2/gross_removals__all_C_pools__MgCO2_ha_yr_2017.tif
  EE asset: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/removals/gross_removals__all_C_pools__MgCO2_ha_yr_2017_test
  EE type:  IMAGE


## Optional code to delete a registered asset:

In [8]:
# assets_to_delete = [
#     "projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017__test",
#     "projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/gross_removals__all_C_pools__MgCO2_ha_yr_2017__test",
# ]
#
# for asset_id in assets_to_delete:
#     print(f"Deleting: {asset_id}")
#     ee.data.deleteAsset(asset_id)

Deleting: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017__test
Deleting: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/gross_removals__all_C_pools__MgCO2_ha_yr_2017__test
